# create sp for logging

In [30]:
use [sample];
GO

SELECT DB_NAME() AS db_name;
GO

Commands completed successfully.

(1 row affected)

db_name
-------
sample 
(1 row)

## spLog - base sp for logging

In [31]:
/*
IF OBJECT_ID('spLog', 'P') IS NOT NULL
BEGIN
    DROP PROCEDURE dbo.spLog
    PRINT 'dbo.spLog DELETED'
END
IF OBJECT_ID('spInfo', 'P') IS NOT NULL
BEGIN
    DROP PROCEDURE dbo.spInfo
    PRINT 'dbo.spInfo DELETED'
END
IF OBJECT_ID('spWarn', 'P') IS NOT NULL
BEGIN
    DROP PROCEDURE dbo.spWarn
    PRINT 'dbo.spWarn DELETED'
END
IF OBJECT_ID('spError', 'P') IS NOT NULL
BEGIN
    DROP PROCEDURE dbo.spError
    PRINT 'dbo.spError DELETED'
END
*/
DROP PROCEDURE IF EXISTS dbo.spLog, dbo.spInfo, dbo.spWarn, dbo.spError;
PRINT 'Cleanup complete.';

Cleanup complete.

In [32]:
CREATE OR ALTER PROCEDURE dbo.spLog
    @Level NVARCHAR(10),    -- failing to specify size defaults to 1 !!!
    @Message NVARCHAR(MAX)
AS
BEGIN
    SELECT
        GETDATE() as [Timestamp],
        @Level as [Level], 
        @Message as [Message];

    DECLARE @Date NVARCHAR(20) = CONVERT(NVARCHAR, GETDATE(), 120) -- Style 120 is the "Golden Standard"
    SET @Level = UPPER(@Level)
    -- PRINT '@Date: ' + @Date;
    -- PRINT '@Level: ' + @Level;
    DECLARE @Msg NVARCHAR(MAX) = FORMATMESSAGE('%s | %s | %s', 
        @Date,
        @Level, 
        @Message
    );
    PRINT @Msg;
END
GO


Commands completed successfully.

In [33]:
EXEC dbo.spLog 'INFO', 'hello';

(1 row affected)
2026-03-25 07:50:05 | INFO | hello

Timestamp               | Level | Message
------------------------+-------+--------
2026-03-25 07:50:05.880 | INFO  | hello  
(1 row)

## add info, warning, error helpers

In [34]:
CREATE OR ALTER PROCEDURE dbo.spInfo
    @Message NVARCHAR(MAX)
AS
BEGIN
    EXEC dbo.spLog 'INFO', @Message;
END
GO


Commands completed successfully.

In [35]:
CREATE OR ALTER PROCEDURE dbo.spWarn
    @Message NVARCHAR(MAX)
AS
BEGIN
    EXEC dbo.spLog 'WARN', @Message;
END
GO


Commands completed successfully.

In [36]:
CREATE OR ALTER PROCEDURE dbo.spError
    @Message NVARCHAR(MAX)
AS
BEGIN
    DECLARE @ErrorMsg NVARCHAR(MAX) = ''
    -- SELECT ERROR_NUMBER(), ERROR_MESSAGE();
    IF ERROR_MESSAGE() IS NOT NULL
        SET @ErrorMsg = FORMATMESSAGE('%s: %i - %s', 
            @Message, 
            ERROR_NUMBER(), 
            ERROR_MESSAGE()
        );
    ELSE
        SET @ErrorMsg = @Message;
    

    EXEC dbo.spLog 'ERROR', @ErrorMsg;
END
GO


Commands completed successfully.

## test sps

In [37]:
SELECT DB_NAME() AS db_name;
GO

EXEC dbo.spLog @Level='INFO', @Message='hello'
EXEC dbo.spInfo @Message='hello'
EXEC dbo.spWarn @Message='hello'
EXEC dbo.spError @Message='hello'
GO


(1 row affected)

db_name
-------
sample 
(1 row)

(1 row affected)
2026-03-25 07:50:05 | INFO | hello
(1 row affected)
2026-03-25 07:50:05 | INFO | hello
(1 row affected)
2026-03-25 07:50:05 | WARN | hello
(1 row affected)
2026-03-25 07:50:05 | ERROR | hello

Timestamp               | Level | Message
------------------------+-------+--------
2026-03-25 07:50:05.920 | INFO  | hello  
(1 row)

Timestamp               | Level | Message
------------------------+-------+--------
2026-03-25 07:50:05.920 | INFO  | hello  
(1 row)

Timestamp               | Level | Message
------------------------+-------+--------
2026-03-25 07:50:05.920 | WARN  | hello  
(1 row)

Timestamp               | Level | Message
------------------------+-------+--------
2026-03-25 07:50:05.920 | ERROR | hello  
(1 row)